In [1]:
from google.colab import files
upload = files.upload()

Saving archive.zip to archive.zip


In [2]:
from google.colab import files
upload = files.upload()

Saving bank-direct-marketing-campaigns.csv to bank-direct-marketing-campaigns.csv


In [3]:
!pip install codecarbon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.8/179.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 6.6 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report
import warnings
import os

import seaborn as sns
sns.set()

# disabling the display of warnings
warnings.filterwarnings('ignore')
from codecarbon import EmissionsTracker
from codecarbon import track_emissions
import scipy.stats as stats

sns.set()

In [28]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = original_data.select_dtypes(include=['object']).columns

label_encoder = LabelEncoder()

for column in categorical_columns:
    original_data[column] = label_encoder.fit_transform(original_data[column])

original_data = original_data.astype(float)
original_data.head()



,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,1.0,1.0,999.0,0.0,1.0,1.1,93.994,-36.4,4.857,5191.0,0.0
1,57.0,7.0,1.0,3.0,1.0,0.0,0.0,1.0,6.0,1.0,1.0,999.0,0.0,1.0,1.1,93.994,-36.4,4.857,5191.0,0.0
2,37.0,7.0,1.0,3.0,0.0,2.0,0.0,1.0,6.0,1.0,1.0,999.0,0.0,1.0,1.1,93.994,-36.4,4.857,5191.0,0.0
3,40.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,6.0,1.0,1.0,999.0,0.0,1.0,1.1,93.994,-36.4,4.857,5191.0,0.0
4,56.0,7.0,1.0,3.0,0.0,0.0,2.0,1.0,6.0,1.0,1.0,999.0,0.0,1.0,1.1,93.994,-36.4,4.857,5191.0,0.0


In [30]:
original_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  float64
 1   job             41188 non-null  float64
 2   marital         41188 non-null  float64
 3   education       41188 non-null  float64
 4   default         41188 non-null  float64
 5   housing         41188 non-null  float64
 6   loan            41188 non-null  float64
 7   contact         41188 non-null  float64
 8   month           41188 non-null  float64
 9   day_of_week     41188 non-null  float64
 10  campaign        41188 non-null  float64
 11  pdays           41188 non-null  float64
 12  previous        41188 non-null  float64
 13  poutcome        41188 non-null  float64
 14  emp.var.rate    41188 non-null  float64
 15  cons.price.idx  41188 non-null  float64
 16  cons.conf.idx   41188 non-null  float64
 17  euribor3m       41188 non-null 

In [8]:
original_data.isnull().sum()

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [ ]:
# define target variable
#y = data["Diabetes_binary"]
# drop target variable/define predictors
#X = data.drop(labels = ["campaign"], axis = 1)
#X.head()

In [32]:
 # define models

clf_rdf = RandomForestClassifier()
clf_dtc = DecisionTreeClassifier()
clf_xgb = xgb.XGBClassifier()
clf_lr = LogisticRegression()
clf_knn = KNeighborsClassifier(n_neighbors=3)
clf_svc = SVC()
clf_gnb = GaussianNB()

In [33]:
 model_track_name = ''

In [34]:
def model_fit(model_name, model, num_features, reduction_percentage, X_train, y_train, run):
    project_name = f"{model_name}_{num_features}_features_reduced_by_{reduction_percentage}_percent{run}"
    print(model_name)

    tracker = EmissionsTracker(project_name=project_name)
    tracker.start()
    model.fit(X_train, y_train)
    tracker.stop()

In [35]:
# csv adaption
def csv_adaption(model_name, model, num_features, reduction_percentage, X_train, y_train, X, run):
    project_name = f"{model_name}_{num_features}_features_reduced_by_{reduction_percentage}_percent{run}"
    columns_names = ', '.join(X.columns)


    if os.path.exists('emissions_data.csv'):
      dataf = pd.read_csv('emissions_data.csv')
      new_row = {'project_name': project_name,'num_features': num_features,'reduction_percentage': reduction_percentage, 'feature_names': columns_names}
      idx = len(dataf)  # Bestimme den Index für die neue Zeile
      dataf.loc[idx] = new_row
      #dataf.append(new_row, ignore_index=True)
      dataf.to_csv('emissions_data.csv', index=False)

    else:
      data ={
        'project_name': [project_name],
        'num_features': [num_features],
        'reduction_percentage': [reduction_percentage],
        'feature_names': [columns_names]
        }
      dataf = pd.DataFrame(data)
      dataf.to_csv('emissions_data.csv', index=False)


In [36]:
# row reduction
def reduce_dataset(dataframe, reduction_percentage):
    num_samples = int(len(dataframe) * (reduction_percentage / 100))
    reduced_dataframe = dataframe.sample(frac=(1 - (num_samples / len(dataframe))))

    return reduced_dataframe

In [37]:
def row_main_variation(data, model_name, model, num_features, run):

    for i in reduction_array:

        print()
        print("Dataset reduced by [%]:", i)

        df_reduced_samples = reduce_dataset(data, i)
        num_rows = df_reduced_samples.shape[0]
        print("Number of rows:", num_rows)

        # feature to be predicted
        y = df_reduced_samples["campaign"]
        #print(y)
        # all other features are used as predictors
        X = df_reduced_samples.drop(labels = ["campaign"], axis = 1)

        # split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=110)

        # scale data
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

        # train models
        model_fit(model_name, model, num_features, i, X_train, y_train, run)
        csv_adaption(model_name, model, num_features, i, X_train, y_train, X, run)

In [38]:
# feature reduction
reduction_array = [0,20,40,60,80]
#clf_names = ['RandomForest', 'DecisionTree', 'XGBoost', 'LogisticRegression', 'KNeighborsClassifier', 'GaussianNB']
#clf_names = ['RandomForest']
#clf_names = ['DecisionTree']
#clf_names = ['XGBoost']
clf_names = ['LogisticRegression']
#clf_names = ['KNeighborsClassifier']
#clf_names = ['GaussianNB']

#clf_list = [clf_rdf, clf_dtc, clf_xgb, clf_lr, clf_knn, clf_gnb]
#clf_list = [clf_rdf]
#clf_list = [clf_dtc]
#clf_list = [clf_xgb]
clf_list = [clf_lr]
#clf_list = [clf_knn]
#clf_list = [clf_gnb]
feature_reduction = [19,18,17,16,15,14,13,12,11,10,9,8,7,6,5,4,3,2,1]
#data = original_data.copy()
i = 0

# all feature
while i <= 9:
  for count, clf in enumerate(clf_list):

      for num in feature_reduction:

          # restore original data
          data = original_data.copy()

          print()
          print("Number of features:", num)


          # drop target variable
          targetvariable = "campaign"
          data_temp = data.copy()
          #data_temp.drop(columns=targetvariable)
          data_temp = data_temp.drop(labels = ["campaign"], axis = 1)
          #print(data_temp)
          # reduce feature number by num (random)
          df_reducedfeature = data_temp.sample(n=num, axis=1)
          # add target variable again
          df_reducedfeature = pd.concat([data[targetvariable], df_reducedfeature], axis=1)


          print(df_reducedfeature.columns)

          # variation
          row_main_variation(df_reducedfeature, clf_names[count], clf, num, i)
  i += 1

# csv merge
df1 = pd.read_csv('emissions.csv')
df2 = pd.read_csv('emissions_data.csv')
df_final = df1.merge(df2, on='project_name', how='outer')
df_final.to_csv('emissions_full.csv', index=False)


[codecarbon INFO @ 18:48:43] [setup] RAM Tracking...
[codecarbon INFO @ 18:48:43] [setup] GPU Tracking...
[codecarbon INFO @ 18:48:43] No GPU found.
[codecarbon INFO @ 18:48:43] [setup] CPU Tracking...
[codecarbon WARNING @ 18:48:43] No CPU tracking mode found. Falling back on CPU constant mode.



Number of features: 19
Index(['campaign', 'previous', 'job', 'education', 'contact', 'pdays',
       'euribor3m', 'housing', 'cons.price.idx', 'age', 'loan',
       'cons.conf.idx', 'month', 'emp.var.rate', 'nr.employed', 'day_of_week',
       'y', 'poutcome', 'default', 'marital'],
      dtype='object')

Dataset reduced by [%]: 0
Number of rows: 41188
LogisticRegression


[codecarbon WARNING @ 18:48:44] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 18:48:44] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:48:44] >>> Tracker's metadata:
[codecarbon INFO @ 18:48:44]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 18:48:44]   Python version: 3.10.12
[codecarbon INFO @ 18:48:44]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 18:48:44]   Available RAM : 12.678 GB
[codecarbon INFO @ 18:48:44]   CPU count: 2
[codecarbon INFO @ 18:48:44]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:48:44]   GPU count: None
[codecarbon INFO @ 18:48:44]   GPU model: None
[codecarbon INFO @ 18:48:51] Energy consumed for RAM : 0.000008 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 18:48:51] Energy consumed for all CPUs : 0.000071 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 18:48:51] 0.000079 kWh of electric


Dataset reduced by [%]: 20
Number of rows: 32951
LogisticRegression


[codecarbon WARNING @ 18:48:52] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 18:48:52] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:48:52] >>> Tracker's metadata:
[codecarbon INFO @ 18:48:52]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 18:48:52]   Python version: 3.10.12
[codecarbon INFO @ 18:48:52]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 18:48:52]   Available RAM : 12.678 GB
[codecarbon INFO @ 18:48:52]   CPU count: 2
[codecarbon INFO @ 18:48:52]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:48:52]   GPU count: None
[codecarbon INFO @ 18:48:52]   GPU model: None
[codecarbon INFO @ 18:48:58] Energy consumed for RAM : 0.000007 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 18:48:58] Energy consumed for all CPUs : 0.000066 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 18:48:58] 0.000074 kWh of electric

...

[codecarbon WARNING @ 18:49:15] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 18:49:15] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:49:15] >>> Tracker's metadata:
[codecarbon INFO @ 18:49:15]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 18:49:15]   Python version: 3.10.12
[codecarbon INFO @ 18:49:15]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 18:49:15]   Available RAM : 12.678 GB
[codecarbon INFO @ 18:49:15]   CPU count: 2
[codecarbon INFO @ 18:49:15]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:49:15]   GPU count: None
[codecarbon INFO @ 18:49:15]   GPU model: None
[codecarbon INFO @ 18:49:22] Energy consumed for RAM : 0.000009 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 18:49:22] Energy consumed for all CPUs : 0.000083 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 18:49:22] 0.000092 kWh of electric


Dataset reduced by [%]: 20
Number of rows: 32951
LogisticRegression


[codecarbon WARNING @ 18:49:24] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon INFO @ 18:49:24] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:49:24] >>> Tracker's metadata:
[codecarbon INFO @ 18:49:24]   Platform system: Linux-5.15.120+-x86_64-with-glibc2.35
[codecarbon INFO @ 18:49:24]   Python version: 3.10.12
[codecarbon INFO @ 18:49:24]   CodeCarbon version: 2.3.1
[codecarbon INFO @ 18:49:24]   Available RAM : 12.678 GB
[codecarbon INFO @ 18:49:24]   CPU count: 2
[codecarbon INFO @ 18:49:24]   CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon INFO @ 18:49:24]   GPU count: None
[codecarbon INFO @ 18:49:24]   GPU model: None
[codecarbon INFO @ 18:49:28] Energy consumed for RAM : 0.000005 kWh. RAM Power : 4.7543792724609375 W
[codecarbon INFO @ 18:49:28] Energy consumed for all CPUs : 0.000048 kWh. Total CPU Power : 42.5 W
[codecarbon INFO @ 18:49:28] 0.000053 kWh of electric


Dataset reduced by [%]: 40
Number of rows: 24713
LogisticRegression


KeyboardInterrupt: ignored